# Step 3. Write, and read back

*With ngff-zarr.*

**Goal:** put the two images and the transformation between them in one OME-Zarr 0.6 file, and check what a reader
finds in it. No registration tool is needed from here on.

In [ ]:
import os

import ngff_zarr as nz
from common import DATA, scene_transformation
from ngff_zarr import CoordinateSystemIdentifier, NgffScene

STORE, SCENE = "../output/pair.ome.zarr", "../output/pair.ozx"
registration = scene_transformation("../output/registration/P000/Transform.ome.zarr", STORE)  # an affine, or a field copied below the scene
registration.input = CoordinateSystemIdentifier(path="fixed", name="intrinsic")  # the scene names what it connects
registration.output = CoordinateSystemIdentifier(path="moving", name="intrinsic")
images = {"fixed": nz.from_ome_zarr(f"{DATA}/fixed.ome.zarr"), "moving": nz.from_ome_zarr(f"{DATA}/moving.ome.zarr")}
NgffScene(images=images, coordinateTransformations=[registration]).to_ome_zarr(STORE, overwrite=False)
nz.write_store_to_zip(STORE, SCENE, version="0.6")  # the same store, zipped into one file (RFC-9)
print(SCENE, os.path.getsize(SCENE) // 1_000_000, "MB")

../output/pair.ozx 41 MB


`pair.ozx` is one file:

```text
pair.ozx
├── zarr.json                      the scene: the transformation, and what it connects
├── fixed/                         the fixed image, with its pyramid
├── moving/                        the moving image, with its pyramid
└── coordinateTransformations/
    └── affine/                    the affine's matrix; a displacements field goes in registration/
```

- **An affine is twelve numbers,** a 3 × 4 matrix over (z, y, x) in micrometers, stored as a small array below the
  scene. A `displacements` field is an OME-Zarr image of vectors, copied below the scene. The scene points to either
  one by its path. `scene_transformation` in [common.py](common.py) handles both; [write.py](write.py) is this cell as
  a script.
- **The scene copies the images, as they are:** 42 MB, all of it the two pyramids, unchanged. What is never written is
  a *resampled* image: the transformation is applied when a reader asks, in step 4.
- **The scene says what it connects:** from the coordinate system of `fixed` to the one of `moving`.

## Read it back

This is what a reader finds: the file's metadata, and the transformation in full.

In [ ]:
import numpy as np

scene = NgffScene.from_ome_zarr(SCENE)
T = scene.coordinateTransformations[0]
print({name: image.images[0].data.shape for name, image in scene.images.items()})
print(T.type, "from", T.input.path, "to", T.output.path)
print(np.array2string(np.asarray(T.affine), precision=3, suppress_small=True) if T.type == "affine" else nz.from_ome_zarr(f"{SCENE}/{T.path}").images[0].data.shape)

{'fixed': (50, 680, 535), 'moving': (50, 680, 535)}
affine from fixed to moving
[[  0.999  -0.013   0.034   0.344]
 [  0.013   1.      0.001 -10.088]
 [ -0.035  -0.001   0.999  37.001]]


The matrix maps a point of the fixed image to the same point of the moving image, in micrometers and in the order
z, y, x: a rotation of a few degrees, and in the last column the translation, 37 µm along x.

In `zarr.json`, at the root of the file, the scene names the transformation and the path of its matrix:

In [ ]:
import json
import zipfile

root = json.loads(zipfile.ZipFile(SCENE).read("zarr.json"))
print(json.dumps(root["attributes"]["ome"]["scene"]["coordinateTransformations"], indent=1))

[
 {
  "input": {
   "path": "fixed",
   "name": "intrinsic"
  },
  "output": {
   "path": "moving",
   "name": "intrinsic"
  },
  "type": "affine",
  "path": "coordinateTransformations/affine"
 }
]


## The size of the transformation

| Preset | What step 2 writes | `Transform.ome.zarr` | `pair.ozx` |
|---|---|---|---|
| `Generic_Rigid` | an `affine`: twelve numbers | 4 kB | 42 MB, the two images |
| `Generic_Rigid_BSpline` | a `displacements` field: float32, on the fixed grid | 157 MB | 196 MB |

The ITK form the preset writes without `Transform:omezarr` tells the same story: an `AffineTransform` of a few
kilobytes, or a 437 MB field.

**Next: [Step 4. Apply](4-apply.ipynb)**